# Hybrid ranking

Semantic, lexical, and code-aware retrieval use different measurements.
Reciprocal Rank Fusion (RRF) combines their rank positions rather than
incorrectly comparing their raw scores.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load an embedding model


In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}


In [ ]:
from qachat.runtime import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")


In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare real repository data


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)


In [ ]:
from qachat.sources.sources import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


## 3. Ask one question

The question names a function. This supplies useful evidence to all three
retrieval channels.


In [ ]:
QUESTION = "How does semantic_search rank repository chunks?"


In [ ]:
from qachat.query_formats import (
    SOFTWARE_REPOSITORY_INSTRUCTION,
    format_qwen3_query,
)

semantic_query = format_qwen3_query(
    question=QUESTION,
    instruction=SOFTWARE_REPOSITORY_INSTRUCTION,
)


## 4. Run each retrieval channel


In [ ]:
from qachat.retrievals import (
    build_semantic_index,
    semantic_search,
)

semantic_index = build_semantic_index(
    chunks,
    embedding_session,
)
semantic_results = semantic_search(
    semantic_index,
    semantic_query,
    embedding_session,
    count=10,
)


In [ ]:
from qachat.retrievals import (
    build_lexical_index,
    lexical_search,
)

lexical_index = build_lexical_index(chunks)
lexical_results = lexical_search(
    lexical_index,
    QUESTION,
    count=10,
)


In [ ]:
from qachat.retrievals import (
    build_code_graph,
    code_search,
)

code_graph = build_code_graph(source_files, units)
code_results = code_search(
    code_graph,
    chunks,
    QUESTION,
    count=10,
)


## 5. Inspect the individual rankings

Before combining them, inspect the top results from each retrieval channel.
This shows how meaning, wording, and code relationships rank the same
repository differently.


In [ ]:
def show_ranking(name, results, count=5):
    print(f"\n{name}")

    for rank, result in enumerate(results[:count], start=1):
        label = result.chunk.symbol or result.chunk.path
        print(f"{rank}. {label}")


show_ranking("Semantic", semantic_results)
show_ranking("Lexical", lexical_results)
show_ranking("Code-aware", code_results)


## 6. Fuse the rankings

The three channels use different raw scores, so those scores should not be
compared directly.

Reciprocal Rank Fusion (RRF) instead uses rank positions. For rank `r`, it
adds `1 / (60 + r)`. A chunk found by several channels receives one
contribution from each channel.


In [ ]:
from qachat.retrievals import hybrid_search

hybrid_results = hybrid_search(
    semantic=semantic_results,
    lexical=lexical_results,
    code=code_results,
    count=5,
)

for rank, result in enumerate(hybrid_results, start=1):
    print(
        f"{rank}. {result.score:.4f} "
        f"{result.channels} "
        f"{result.chunk.symbol}"
    )


## 7. Inspect how RRF combined them

First convert each retrieval list into a simple mapping from chunk identifier
to rank position.


In [ ]:
def rank_positions(results):
    return {
        result.chunk.identifier: rank
        for rank, result in enumerate(results, start=1)
    }


semantic_ranks = rank_positions(semantic_results)
lexical_ranks = rank_positions(lexical_results)
code_ranks = rank_positions(code_results)


Now compare where each hybrid result appeared in the three original
rankings.


In [ ]:
print(
    f"{'Chunk':<32}"
    f"{'Semantic':>10}"
    f"{'Lexical':>10}"
    f"{'Code':>10}"
    f"{'RRF':>12}"
)

for result in hybrid_results:
    identifier = result.chunk.identifier
    name = result.chunk.symbol or result.chunk.path.name

    print(
        f"{str(name):<32}"
        f"{str(semantic_ranks.get(identifier, '-')):>10}"
        f"{str(lexical_ranks.get(identifier, '-')):>10}"
        f"{str(code_ranks.get(identifier, '-')):>10}"
        f"{result.score:>12.4f}"
    )


A result does not need to rank first in any single channel to become the
best hybrid result. RRF rewards chunks that several independent retrieval
methods rank highly.


## 8. Inspect the best evidence

The highest-ranked hybrid result is the chunk RRF considers strongest after
combining the three retrieval channels.


In [ ]:
best = hybrid_results[0]

print(f"Best source: {best.chunk.path}")
print(f"Channels: {best.channels}")
print()
print(best.chunk.text)


## 9. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Result

QAChat combined meaning, wording, and code relationships into one ranking.
RRF kept the different raw scoring systems separate and promoted chunks that
ranked well across multiple retrieval channels.
